# No.2 1D FFT / IFFT

矩形パルス（kukei.ipynb で生成）に FFT をかけて周波数スペクトルを確認します。

`flag = 0` で FFT、`flag = 1` で IFFT を実行します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io


In [ ]:
# --- パラメータ ---
input_filename = 'kukei_DW17.mat'  # kukei.ipynb で生成したファイル
flag = 0  # 0: FFT,  1: IFFT

# データ読み込み
data = scipy.io.loadmat(input_filename)
keys = [k for k in data.keys() if not k.startswith('_')]
Signal = data[keys[0]].flatten().astype(complex)
print(f'読み込み: {input_filename}, shape={Signal.shape}')

In [ ]:
# FFT / IFFT
if flag == 0:
    Output = np.fft.fftshift(np.fft.fft(np.fft.fftshift(Signal)))
    title_prefix = 'FFT'
else:
    Output = np.fft.fftshift(np.fft.ifft(np.fft.fftshift(Signal)))
    title_prefix = 'IFFT'

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(Output.real, linewidth=1, label='Real')
axes[0].set_title(f'{title_prefix} — Real part  ← sinc 関数になっているか確認')
axes[0].legend()
axes[0].grid(True)
axes[1].plot(Output.imag, linewidth=1, label='Imaginary', color='orange')
axes[1].set_title(f'{title_prefix} — Imaginary part  ← 0 に近いはず（実信号なので）')
axes[1].legend()
axes[1].grid(True)
fig.tight_layout()
plt.show()

# 保存
out_name = input_filename.replace('.mat', f'_{title_prefix}.mat')
scipy.io.savemat(out_name, {'Signal': Output})
print(f'{out_name} を保存しました')

## 窓幅の違いを比較

`DW=17` と `DW=65` の FFT 結果を並べて比較します。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, dw in enumerate([17, 65]):
    try:
        d = scipy.io.loadmat(f'kukei_DW{dw}.mat')
        sig = d[[k for k in d if not k.startswith('_')][0]].flatten().astype(complex)
        fft_out = np.fft.fftshift(np.fft.fft(np.fft.fftshift(sig)))
        axes[0, col].plot(sig.real, linewidth=1)
        axes[0, col].set_title(f'Rectangular pulse (DW={dw})')
        axes[0, col].grid(True)
        axes[1, col].plot(fft_out.real, linewidth=1)
        axes[1, col].set_title(f'FFT real part (DW={dw})')
        axes[1, col].grid(True)
    except FileNotFoundError:
        axes[0, col].set_title(f'kukei_DW{dw}.mat が見つかりません')

fig.tight_layout()
plt.show()
print('DW が大きい（パルスが広い）ほど FFT のメインローブは狭くなる')